<a href="https://colab.research.google.com/github/lahari600/Flyrank/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lahari600/Flyrank/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule:
I rank content based on search performance. Pages with higher Google Search impressions, clicks, and organic sessions receive a higher baseline score and are recommended for content refresh.

Reason Codes:
• HIGH_IMPRESSIONS
• HIGH_CLICKS
• HIGH_ORGANIC_TRAFFIC

Action Label:
Refresh Content

In [8]:

!pip -q install datasets huggingface_hub pyarrow

from datasets import load_dataset
from google.colab import userdata
import pandas as pd

token = userdata.get("HF_TOKEN")

dataset = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    streaming=True,
    token=token
)

df = pd.DataFrame(list(dataset.take(1000)))

print(df.shape)
df.head()

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

(1000, 30)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,0
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,0
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,0
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,0
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,0


In [3]:

import pandas as pd

df["baseline_score"] = (
    0.5 * df["gsc_impressions"] +
    0.3 * df["gsc_clicks"] +
    0.2 * df["sessions_organic"]
)

df["reason_code"] = "HIGH_IMPRESSIONS"
df["action"] = "Refresh Content"

df[[
    "content_hash_id",
    "baseline_score",
    "reason_code",
    "action"
]].head()

,content_hash_id,baseline_score,reason_code,action
0,content_3b70a18ea133b2bb,15.0,HIGH_IMPRESSIONS,Refresh Content
1,content_fe8e8155ce1d47a2,2.5,HIGH_IMPRESSIONS,Refresh Content
2,content_b4462a1b90640058,0.5,HIGH_IMPRESSIONS,Refresh Content
3,content_c899aef92518c714,3.0,HIGH_IMPRESSIONS,Refresh Content
4,content_c7c1d2e68d9d0964,2.5,HIGH_IMPRESSIONS,Refresh Content


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

The baseline score combines impressions, clicks, and organic sessions. The ranked list is written to work/outputs/baseline_action_score.csv.

In [4]:

import os

# Create a simple baseline score
df["baseline_score"] = (
    0.5 * df["gsc_impressions"] +
    0.3 * df["gsc_clicks"] +
    0.2 * df["sessions_organic"]
)

# Reason code
df["reason_code"] = "HIGH_TRAFFIC"

# Action
df["action"] = "Improve SEO"

# Rank
ranked = df.sort_values("baseline_score", ascending=False)

# Save CSV
os.makedirs("work/outputs", exist_ok=True)

ranked.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("CSV saved successfully!")
ranked.head(10)

CSV saved successfully!


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,baseline_score,reason_code,action
446,2025-01-28,client_9958f0a7ae1df715,content_f94fe855380e150f,True,True,True,False,303,3,549,...,0,0,0,0,0,0,0,152.4,HIGH_TRAFFIC,Improve SEO
146,2025-01-27,client_9958f0a7ae1df715,content_f94fe855380e150f,True,True,True,False,266,6,482,...,0,0,0,0,0,0,0,134.8,HIGH_TRAFFIC,Improve SEO
766,2025-01-29,client_9958f0a7ae1df715,content_f94fe855380e150f,True,True,True,False,240,5,434,...,0,0,0,0,0,0,0,121.5,HIGH_TRAFFIC,Improve SEO
953,2025-01-30,client_9958f0a7ae1df715,content_37b3bafd5f88fdd1,True,True,True,False,202,4,1017,...,0,0,0,0,0,0,0,102.2,HIGH_TRAFFIC,Improve SEO
349,2025-01-28,client_9958f0a7ae1df715,content_d02be57d816cf3d7,True,True,True,False,174,1,525,...,0,0,0,0,0,0,0,87.3,HIGH_TRAFFIC,Improve SEO
45,2025-01-27,client_9958f0a7ae1df715,content_d02be57d816cf3d7,True,True,True,False,118,0,406,...,0,0,0,0,0,0,0,59.0,HIGH_TRAFFIC,Improve SEO
665,2025-01-29,client_9958f0a7ae1df715,content_d02be57d816cf3d7,True,True,True,False,97,0,393,...,0,0,0,0,0,0,0,48.5,HIGH_TRAFFIC,Improve SEO
954,2025-01-30,client_9958f0a7ae1df715,content_de7b08874af74c00,True,True,True,False,87,0,623,...,0,0,0,0,0,0,0,43.5,HIGH_TRAFFIC,Improve SEO
699,2025-01-29,client_9958f0a7ae1df715,content_84f5a9ecfefa108e,True,True,True,False,81,1,202,...,0,0,0,0,0,0,0,40.8,HIGH_TRAFFIC,Improve SEO
76,2025-01-27,client_9958f0a7ae1df715,content_de7b08874af74c00,True,True,True,False,80,1,481,...,0,0,0,0,0,0,0,40.3,HIGH_TRAFFIC,Improve SEO


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

The top-ranked pages were reviewed manually. Recommendations are based on current traffic signals. They may be incorrect if the traffic is seasonal or the pages are already optimized.

In [6]:

top20 = ranked.head(20)

for i, row in top20.iterrows():
    print(f"{i+1}.")
    print("Content:", row["content_hash_id"])
    print("Action: Improve SEO")
    print("Reason Code: HIGH_TRAFFIC")
    print("Confidence: High traffic based on impressions and organic sessions.")
    print("What would make it wrong: Traffic may be seasonal or the page may already be optimized.")
    print("-"*60)

447.
Content: content_f94fe855380e150f
Action: Improve SEO
Reason Code: HIGH_TRAFFIC
Confidence: High traffic based on impressions and organic sessions.
What would make it wrong: Traffic may be seasonal or the page may already be optimized.
------------------------------------------------------------
147.
Content: content_f94fe855380e150f
Action: Improve SEO
Reason Code: HIGH_TRAFFIC
Confidence: High traffic based on impressions and organic sessions.
What would make it wrong: Traffic may be seasonal or the page may already be optimized.
------------------------------------------------------------
767.
Content: content_f94fe855380e150f
Action: Improve SEO
Reason Code: HIGH_TRAFFIC
Confidence: High traffic based on impressions and organic sessions.
What would make it wrong: Traffic may be seasonal or the page may already be optimized.
------------------------------------------------------------
954.
Content: content_37b3bafd5f88fdd1
Action: Improve SEO
Reason Code: HIGH_TRAFFIC
Confidenc

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks have very low impressions and clicks, so they receive low scores. Only current-day signals were used, and no future data or labels were included

In [7]:

weak = ranked.tail(10)

print("Weak Picks")
display(weak[[
    "content_hash_id",
    "baseline_score",
    "gsc_impressions",
    "gsc_clicks",
    "sessions_organic"
]])

print("\nLeakage Check")
print("✓ Only current-day features were used.")
print("✓ No future information was used.")
print("✓ No labels or target variables were used.")
print("✓ Baseline score is based only on available signals.")

Weak Picks


,content_hash_id,baseline_score,gsc_impressions,gsc_clicks,sessions_organic
132,content_ead0d09afb9004d0,0.5,1,0,0
114,content_c99ba71d9e71ac66,0.5,1,0,0
55,content_5175438fecb054a4,0.5,1,0,0
62,content_1c929a926abb6495,0.5,1,0,0
64,content_6c845eb06ed5c8a8,0.5,1,0,0
67,content_17c76438024dece9,0.5,1,0,0
48,content_a6f74f9c4e58d4f0,0.5,1,0,0
24,content_d0d52c7ff7217dac,0.5,1,0,0
968,content_bf2e0c4a8b71f5ef,0.5,1,0,0
2,content_b4462a1b90640058,0.5,1,0,0



Leakage Check
✓ Only current-day features were used.
✓ No future information was used.
✓ No labels or target variables were used.
✓ Baseline score is based only on available signals.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.